# A1 · 診斷決策系統 — 從後驗到「該不該治療」

> **核心問題**：模型輸出一個機率（例如「90% 是惡性」）。**把它變成行動的那一步，數學長什麼樣？**

資料：Heart Disease (Cleveland)，A-D1（297 筆、13 臨床特徵、二元診斷）。
本 notebook 走：貝葉斯邏輯迴歸 → 後驗 → 後驗預測 vs plug-in → 損失矩陣與最優門檻 → 棄權選項。
核心程式在 [`../src/`](../src)。

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
import data, bayes_logreg as blr, decision as dec
DATA = os.path.abspath('../../data/A_medical')
ds = data.load_heart(DATA, test_size=0.25, seed=0)
idata, _ = blr.fit(ds.X_train, ds.y_train, prior_sd=2.5, seed=42)
print('特徵數', ds.n_features, '| train', len(ds.y_train), '| test', len(ds.y_test))
print('收斂', blr.convergence(idata))

Initializing NUTS using jitter+adapt_diag...


Multiprocess sampling (4 chains in 4 jobs)


NUTS: [beta, alpha]


Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 2 seconds.


特徵數 18 | train 222 | test 75
收斂 {'max_rhat': 1.0, 'min_ess_bulk': 3923.0}


## 1 · 貝葉斯邏輯迴歸（弱資訊先驗）

$\beta \sim \mathcal{N}(0, 2.5)$、$\alpha \sim \mathcal{N}(0, 5)$、$y \sim \text{Bernoulli}(\text{logit}=\alpha+X\beta)$。
N(0, 2.5) 是**標準化後**邏輯迴歸的標準弱資訊先驗（計劃書主題二）——所以我們先把連續特徵標準化。

In [2]:
auc_b = roc_auc_score(ds.y_test, blr.predictive_mean(idata, ds.X_test))
auc_f = roc_auc_score(ds.y_test, LogisticRegression(max_iter=2000)
                      .fit(ds.X_train, ds.y_train).predict_proba(ds.X_test)[:,1])
print(f'測試 AUC：貝葉斯={auc_b:.3f} · 頻率派={auc_f:.3f}')

測試 AUC：貝葉斯=0.908 · 頻率派=0.907


## 2 · 後驗：誰驅動心臟病風險？

貝葉斯給的不是一組點估計，而是每個係數的**完整後驗分佈**（帶不確定性）。

![係數森林圖](../figures/01_coefficients.png)

醫學上合理：`thal_7`（可逆缺損）、`sex`（男性）、`ca`（造影血管數）是正風險因子；
`thalach`（最大心率）是保護因子。類別稀少的 `restecg_1`/`slope_3` 可信區間很寬——**誠實的不確定性**。

## 3 · 後驗預測分佈 vs plug-in（步驟 2）

- **plug-in**：用後驗均值參數做「一次」預測 → $\text{sigmoid}(\mathbb{E}[\alpha]+X\mathbb{E}[\beta])$
- **後驗預測**：用「全部後驗樣本」各預測一次再平均 → $\mathbb{E}[\text{sigmoid}(\alpha+X\beta)]$

因 sigmoid 非線性（Jensen 不等式），兩者不同：後驗預測被不確定性**拉向 0.5**，而且每個病人帶一條可信區間。

In [3]:
p_pred = blr.predictive_mean(idata, ds.X_test)
p_plug = blr.plugin_proba(idata, ds.X_test)
lo, hi = blr.predictive_interval(idata, ds.X_test)
i = int(np.abs(p_plug - p_pred).argmax())
print(f'最大差異病人：plug-in={p_plug[i]:.3f}  後驗預測={p_pred[i]:.3f}  95%CI=[{lo[i]:.3f}, {hi[i]:.3f}]')
print(f'後驗預測 95%CI 平均寬度={np.mean(hi-lo):.3f}（plug-in 寬度=0）')

最大差異病人：plug-in=0.845  後驗預測=0.798  95%CI=[0.371, 0.980]
後驗預測 95%CI 平均寬度=0.346（plug-in 寬度=0）


> **費曼檢驗：後驗預測和 plug-in 差在哪個具體數字上？**
> 這位病人 plug-in 自信地說 0.845，後驗預測說 0.798——更重要的是它附了 95%CI ≈ [0.37, 0.98]，
> 意思是「可能從 37% 到 98%」。**plug-in 把這個不確定性藏起來了。**

![後驗預測 vs plug-in](../figures/02_predictive_vs_plugin.png)

## 4 · 損失矩陣與最優門檻（步驟 3）

|          | 有病 y=1 | 沒病 y=0 |
|---|:-:|:-:|
| **治療** | 0 | $C_{FP}$ |
| **不治療** | $C_{FN}$ | 0 |

$\mathbb{E}[\text{loss}\mid\text{治療}]=(1-p)C_{FP}$、$\mathbb{E}[\text{loss}\mid\text{不治療}]=p\,C_{FN}$
$\Rightarrow$ 治療 iff $p > \dfrac{C_{FP}}{C_{FP}+C_{FN}} =: p^*$。**門檻由成本比決定，不是預設的 0.5。**

In [4]:
for r in [1, 5, 20, 100]:
    print(f'C_FN:C_FP = {r:>3}:1  →  最優門檻 p* = {dec.optimal_threshold(1, r):.4f}')
loss_half = dec.realized_loss(ds.y_test, p_pred, 1, 20, 0.5)
ps = dec.optimal_threshold(1, 20)
loss_star = dec.realized_loss(ds.y_test, p_pred, 1, 20, ps)
print(f'\nC_FN:C_FP=20:1 測試集實際損失：門檻0.5={loss_half:.2f}  vs  最優p*={ps:.3f}→{loss_star:.2f}'
      f'  （降低 {1-loss_star/loss_half:.0%}）')

C_FN:C_FP =   1:1  →  最優門檻 p* = 0.5000
C_FN:C_FP =   5:1  →  最優門檻 p* = 0.1667
C_FN:C_FP =  20:1  →  最優門檻 p* = 0.0476
C_FN:C_FP = 100:1  →  最優門檻 p* = 0.0099

C_FN:C_FP=20:1 測試集實際損失：門檻0.5=2.23  vs  最優p*=0.048→0.31  （降低 86%）


> **費曼檢驗：怎麼對非技術背景的人解釋「為什麼門檻不是 0.5」？**
> 「如果漏掉一個病人的代價是誤報的 100 倍，那我寧可寧枉勿縱——只要有 1% 的機率有病就該進一步處理，
> 而不是等到 50% 才行動。」門檻 0.5 只在『兩種錯一樣糟』時才對。

![最優門檻](../figures/03_optimal_threshold.png)

100:1 時最優門檻掉到 **0.01**（和 0.5 差兩個數量級）；用 0.5 當門檻的實際損失是最優的 **7 倍**。

## 5 · 加入棄權選項（步驟 4）

第三個行動「轉診 / 再檢查」，固定成本 $C_{reject}$。當治療與不治療的期望損失**都**高於 $C_{reject}$ 時，就該棄權。

In [5]:
for c_rej in [0.6, 0.4]:
    reg = dec.reject_region(1, 5, c_rej)
    _, frac = dec.realized_loss_with_reject(ds.y_test, p_pred, 1, 5, c_rej)
    print(f'C_reject={c_rej}: 棄權區 p∈[{reg[0]:.2f}, {reg[1]:.2f}]（寬 {reg[1]-reg[0]:.2f}），'
          f'測試集 {frac:.0%} 病人被轉診')

C_reject=0.6: 棄權區 p∈[0.12, 0.40]（寬 0.28），測試集 25% 病人被轉診
C_reject=0.4: 棄權區 p∈[0.08, 0.60]（寬 0.52），測試集 35% 病人被轉診


![棄權選項](../figures/04_reject_option.png)

棄權成本越小 → 棄權區越寬（越傾向「不確定就轉診」）。這把「模型不確定的病人自動導向人工複核」變成一條可計算的規則。

## 重點

1. 把機率變成行動的那一步，就是**期望損失最小化**：治療 iff $p>p^*=C_{FP}/(C_{FP}+C_{FN})$。
2. 門檻由**成本比**決定，醫療上 $C_{FN}\gg C_{FP}$ → 門檻遠低於 0.5。
3. 後驗預測分佈比 plug-in 誠實：它帶著每個病人的不確定性。
4. 棄權選項把「不確定就轉人工」制度化。

→ 但機率要能拿來做決策，得先**校準**。見 [`02_calibration_and_sensitivity.ipynb`](02_calibration_and_sensitivity.ipynb)。